In [5]:
from ultralytics import YOLO
import time as t
import os
import numpy as np
from ultralytics import YOLO
from glob import glob
import cv2
import json
import yaml
import tqdm as tqdm
from scale_bar_detection.scale_bar_detection_utils import detect_scale_bar
import easyocr

DATASET_NAME = "HB-complete-scalebars-noleg-noHwing"

#  MODEL_PATH = "./runs/pose/train/weights/best.pt"
TEST_IMAGES = f"./datasets/{DATASET_NAME}/images/test/"
TEST_LABELS = f"./datasets/{DATASET_NAME}/labels/test/"

IMG_SIZE = 640
CONF = 0.25
DIST_THRESHOLD = 0.05  # seuil normalisé (5% de la bbox)

SCALE_BAR_MODEL_PATH = "./scale_bar_detection/best.pt"

all_kp = [
    "head-top", "head-left","head-right","left-eye","right-eye","neck","thorax-left","thorax-right","thorax-bottom","body-left","body-right",
    "body-tip","left-antenna-0","left-antenna-1","left-antenna-2","right-antenna-0","right-antenna-1","right-antenna-2","left-forewing-base","left-forewing-tip","left-forewing-front",
    "left-forewing-rear","right-forewing-base","right-forewing-tip","right-forewing-front","right-forewing-rear","left-hindwing-base","left-hindwing-tip","left-hindwing-front","left-hindwing-rear",
    "right-hindwing-base","right-hindwing-tip","right-hindwing-front","right-hindwing-rear","left-leg-0","left-leg-1","left-leg-2","left-leg-3","right-leg-0","right-leg-1","right-leg-2","right-leg-3",
]

loaded_model = None

# model = YOLO(MODEL_PATH)

def load_yolo_pose_label(label_path):
    """
    Format YOLO pose :
    class cx cy w h kpt1_x kpt1_y v1 kpt2_x kpt2_y v2 ...
    """
    with open(label_path, "r") as f:
        line = f.readline().strip().split()

    values = list(map(float, line))

    # bbox
    cx, cy, w, h = values[1:5]

    # keypoints
    kpts = np.array(values[5:]).reshape(-1, 3)[:, :2]  # (x, y)

    return np.array([cx, cy, w, h]), kpts


def denormalize(kpts, img_w, img_h):
    kpts[:, 0] *= img_w
    kpts[:, 1] *= img_h
    return kpts

def denormalize_bbox(bbox_norm, img_w, img_h):
    """bbox_norm = [cx, cy, w, h] normalized → [x, y, w, h] in pixels (x/y = top-left)"""
    cx, cy, bw, bh = bbox_norm
    return np.array([
        (cx - bw / 2) * img_w,
        (cy - bh / 2) * img_h,
        bw * img_w,
        bh * img_h,
    ])

def bbox_size_pixels(bbox, img_w, img_h):
    _, _, w, h = bbox
    return np.sqrt((w * img_w) * (h * img_h))


def check_keywords(name, kws):
    count = 0
    for kw in kws:
        if kw in name :
            count += 1
    return count == len(kws)

def incremement_name(base_name:str, existing:list[str]):
    nums = []
    for e in existing:
        if base_name in e:
            nums.append(e.replace(base_name, ""))
    i = 1
    while str(i) in nums:
        i +=1
        if i > 100:
            print("too much existing, return None")
            return
    return base_name+str(i)

In [5]:
base_model = "yolo26s-pose"
params = {
"img" : 400,
"pose": 98.0, # base value = 12.0
"kws": "500epochs-noleg-noHwing",
}

model = base_model + "_" + "_".join([param + str(value) for param, value in params.items()])

with open("run_results.json") as f:
    all_models = json.load(f)

if model in all_models:
    model = model + "-n"

all_models[model] = {"name" : f"{base_model}.pt", "training time" : 0, "keypoints perf":[], "metrics" : []}

print(f"\nTraining {model}")
loaded_model = YOLO(all_models[model]["name"])

T1 = t.time()
config_file = f"./datasets/{DATASET_NAME}/yolo-config.yaml"
loaded_model
with open(config_file, "r") as f:
    data = yaml.safe_load(f)
    current_kp = data["kpt_names"][0]

results_yolo = loaded_model.train(
    data        = config_file, 
    degrees     = 180,
    epochs      = 500, 
    imgsz       = 640,
    pose        = params["pose"])

all_models[model]['training time'] = t.time() - T1



Training yolo26s-pose_img400_pose98.0_kws500epochs-noleg-noHwing
New https://pypi.org/project/ultralytics/8.4.41 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.33 🚀 Python-3.12.3 torch-2.11.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4060, 7805MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./datasets/HB-complete-scalebars-noleg-noHwing/yolo-config.yaml, degrees=180, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=500, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max